## Dataset
### Point to Dataset

In [1]:
import os

#base_path = "/content/3DCNN-Tennis-Action-Recognition/tennis_dataset"
base_path = r"C:\Users\Mini-Berlink\Documents\UNT\26-Spring-CSCE5218\Assignment\Project\3DCNN-Tennis-Action-Recognition-main\tennis_dataset"
class_names = sorted(os.listdir(base_path))

print(class_names)

#Original Image Size = (480,852) #HEIGHT, WIDTH
HEIGHT, WIDTH = 180, 320 # Resize to this dimension


['b_slice', 'b_volley', 'backhand', 'f_volley', 'forehand', 'serve', 'smash']


### Load All action frames into Dataset

In [2]:
import os
import numpy as np
import cv2
import random

# User-defined parameters
F_STEP = 5 # Step size for frame collection
VIDEO_NUM_PER_ACTION = 52 # Number of videos to process per action category

actions = os.listdir(base_path)

# Initialize lists to store sequences of frames per video and their labels
all_video_sequences = []
all_video_labels = []

for action in actions:
    action_path = base_path + "/" + action

    # Assign a unique numerical label for the current action
    current_label = class_names.index(action)
    print(f"Label: {current_label}")

    videos = [f for f in os.listdir(action_path) if f.endswith('.mp4')]
    random.shuffle(videos)
    counter = 1

    for video in videos:
      # Stop processing videos for this action if we've reached the limit
      if counter > VIDEO_NUM_PER_ACTION:
          break

      video_path = action_path + "/" + video
      #print(f"Processing: {video_path}")

      cap = cv2.VideoCapture(video_path)
      total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

      if total_frames == 0:
          cap.release()
          print(f"Skipping empty video: {video_path}")
          continue

      middle_frame = total_frames // 2
      frame_indices_set = set()

      # Add middle frame if valid
      if 0 <= middle_frame < total_frames:
          frame_indices_set.add(middle_frame)

      # Go backwards from middle frame
      current_frame_b = middle_frame - F_STEP
      while current_frame_b >= 0:
          frame_indices_set.add(current_frame_b)
          current_frame_b -= F_STEP

      # Go forwards from middle frame
      current_frame_f = middle_frame + F_STEP
      while current_frame_f < total_frames:
          frame_indices_set.add(current_frame_f)
          current_frame_f += F_STEP

      frame_numbers = sorted(list(frame_indices_set)) # These are 0-indexed frame numbers
      #print(f"frame numbers: {frame_numbers}")

      current_video_frames = []
      for frame_number_0_indexed in frame_numbers:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number_0_indexed)
        ret, frame = cap.read()
        if ret:
          frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
          frame = cv2.resize(frame, (WIDTH, HEIGHT))
          frame = np.array(frame).astype(np.float32) # Ensure consistent dtype
          frame = frame / 255.0 # Normalize pixel values to [0, 1] here
          current_video_frames.append(frame)
        else:
          print(f"Frame {frame_number_0_indexed} not found in {video_path}")

      cap.release()

      if current_video_frames: # Only add if frames were successfully extracted
          all_video_sequences.append(np.array(current_video_frames)) # Store as a NumPy array
          all_video_labels.append(current_label)
      else:
          print(f"No frames extracted for {video_path}")

      counter += 1

# Convert lists of sequences and labels to numpy arrays
# X_full_clips will be an array of objects, where each object is a (sequence_length, H, W, C) numpy array
X_full_clips = np.array(all_video_sequences, dtype=object)
y_full_labels = np.array(all_video_labels)

print(f"Total videos processed: {len(X_full_clips)}")
print(f"Shape of X_full_clips (array of variable-length sequences): {X_full_clips.shape}")
print(f"Shape of y_full_labels: {y_full_labels.shape}")

# Individual clip label and shape:
#for i in range(len(X_full_clips)):
#    print(f"Clip {i}: label={y_full_labels[i]}, shape={X_full_clips[i].shape}")



Label: 2
Label: 0
Label: 1
Label: 4
Label: 3
Label: 5
Label: 6
Total videos processed: 364
Shape of X_full_clips (array of variable-length sequences): (364,)
Shape of y_full_labels: (364,)


### Split the dataset into Train / Val / Test

We split the dataset into Train/ Validation/ Test

In [3]:
from sklearn.model_selection import train_test_split

TRAIN_PCT = 0.7
TEST_PCT = 0.15
VALI_PCT = 1-TRAIN_PCT-TEST_PCT

# Use X_full_clips (array of variable-length video sequences) and y_full_labels as X and y
X = X_full_clips
y = y_full_labels

# first split: train () + temp ()
X_train_videos, X_temp_videos, y_train_videos, y_temp_videos = train_test_split(
    X, y, test_size=1-TRAIN_PCT, random_state=42, stratify=y
)

# second split: val () + test ()
X_val_videos, X_test_videos, y_val_videos, y_test_videos = train_test_split(
    X_temp_videos, y_temp_videos, test_size=TEST_PCT/(TEST_PCT+VALI_PCT), random_state=42, stratify=y_temp_videos
)

print("Train video clips:", len(X_train_videos))
for i in range(len(X_train_videos)):
    print(f"Clip {i}: label={y_train_videos[i]}, shape={X_train_videos[i].shape}")

print("Val video clips:", len(X_val_videos))
for i in range(len(X_val_videos)):
    print(f"Clip {i}: label={y_val_videos[i]}, shape={X_val_videos[i].shape}")

print("Test video clips:", len(X_test_videos))
for i in range(len(X_test_videos)):
    print(f"Clip {i}: label={y_test_videos[i]}, shape={X_test_videos[i].shape}")

# At this point, X_train_videos, X_val_videos, X_test_videos are numpy arrays
# where each element is a numpy array representing a sequence of frames for one video.
# y_train_videos, y_val_videos, y_test_videos are numpy arrays of corresponding labels.


Train video clips: 254
Clip 0: label=1, shape=(13, 180, 320, 3)
Clip 1: label=5, shape=(13, 180, 320, 3)
Clip 2: label=6, shape=(13, 180, 320, 3)
Clip 3: label=2, shape=(13, 180, 320, 3)
Clip 4: label=5, shape=(12, 180, 320, 3)
Clip 5: label=4, shape=(13, 180, 320, 3)
Clip 6: label=5, shape=(12, 180, 320, 3)
Clip 7: label=0, shape=(13, 180, 320, 3)
Clip 8: label=3, shape=(13, 180, 320, 3)
Clip 9: label=4, shape=(13, 180, 320, 3)
Clip 10: label=2, shape=(13, 180, 320, 3)
Clip 11: label=6, shape=(13, 180, 320, 3)
Clip 12: label=1, shape=(13, 180, 320, 3)
Clip 13: label=0, shape=(13, 180, 320, 3)
Clip 14: label=3, shape=(13, 180, 320, 3)
Clip 15: label=5, shape=(13, 180, 320, 3)
Clip 16: label=0, shape=(13, 180, 320, 3)
Clip 17: label=5, shape=(13, 180, 320, 3)
Clip 18: label=5, shape=(13, 180, 320, 3)
Clip 19: label=4, shape=(13, 180, 320, 3)
Clip 20: label=3, shape=(13, 180, 320, 3)
Clip 21: label=6, shape=(13, 180, 320, 3)
Clip 22: label=3, shape=(13, 180, 320, 3)
Clip 23: label=1, sha

## Group Raw Frames into Clips for 3D-CNN Input

For a 3D-CNN, input data typically requires a temporal dimension, meaning individual frames need to be grouped into sequences or 'clips'. This step will transform our `(num_frames, HEIGHT, WIDTH, channels)` data into `(num_clips, SEQUENCE_LENGTH, HEIGHT, WIDTH, channels)`.

### Note on Clip Formation for 3D-CNN Input

In the updated data loading step (Cell `nbOb3GKNMMA-`), frames for each video are now collected into **variable-length sequences** (clips) directly. This ensures that all frames within a given sequence originate from the same video.

Consequently, the explicit `create_clips` function from the previous version of this cell is no longer necessary.

For 3D-CNN models, which typically expect a fixed `SEQUENCE_LENGTH` for batching, you will need to implement a strategy to handle these variable-length video sequences. Common approaches include:

*   **Padding**: Add dummy frames (e.g., zeros) to shorter sequences to match the maximum or a predefined `SEQUENCE_LENGTH`.
*   **Truncation**: Cut longer sequences to a predefined `SEQUENCE_LENGTH`.
*   **Sampling**: Randomly sample a fixed number of frames from each video sequence.

This preprocessing step (padding/truncation) should be applied *after* the train/validation/test split, typically within a custom `tf.data.Dataset` or PyTorch `DataLoader` pipeline, to prepare batches for your 3D-CNN model.

In [4]:
from tensorflow import keras

# Preprocessing Video clips
SEQUENCE_LENGTH = 13 # Number of frames per video clip
NUM_CLASSES = len(class_names) # Get from earlier cell
print(f"Num of Classes : {NUM_CLASSES}")

# Function to pad/truncate video sequences to a fixed length
def preprocess_video_sequences(video_sequences, target_length, height, width, channels):
    processed_sequences = []
    for seq in video_sequences:
        if seq.shape[0] > target_length:
            # Truncate if sequence is longer than target_length
            processed_seq = seq[:target_length]
        elif seq.shape[0] < target_length:
            # Pad with zeros if sequence is shorter
            padding_needed = target_length - seq.shape[0]
            padding_block = np.zeros((padding_needed, height, width, channels), dtype=seq.dtype)
            processed_seq = np.concatenate((seq, padding_block), axis=0)
        else:
            processed_seq = seq
        processed_sequences.append(processed_seq)
    return np.array(processed_sequences)

# Get dimensions from the first video clip (assuming it exists and has the correct shape)
if len(X_train_videos) > 0:
    # X_train_videos contains variable-length arrays, so we take the shape of the frames within the first video.
    # The shape is (num_frames, HEIGHT, WIDTH, CHANNELS) for an individual video.
    _, f_HEIGHT, f_WIDTH, f_CHANNELS = X_train_videos[0].shape
else:
    # Fallback if no videos are present, although unlikely if data loaded correctly
    f_HEIGHT, f_WIDTH, f_CHANNELS = HEIGHT, WIDTH, 3 # Default to expected image size

print(f"Original dimensions per frame: H={f_HEIGHT}, W={f_WIDTH}, C={f_CHANNELS}")

# Preprocess (pad/truncate) video sequences
print("Preprocessing training videos...")
X_train_padded = preprocess_video_sequences(X_train_videos, SEQUENCE_LENGTH, f_HEIGHT, f_WIDTH, f_CHANNELS)
print("Preprocessing validation videos...")
X_val_padded = preprocess_video_sequences(X_val_videos, SEQUENCE_LENGTH, f_HEIGHT, f_WIDTH, f_CHANNELS)
print("Preprocessing test videos...")
X_test_padded = preprocess_video_sequences(X_test_videos, SEQUENCE_LENGTH, f_HEIGHT, f_WIDTH, f_CHANNELS)

print(f"Shape of X_train_padded: {X_train_padded.shape}")
print(f"Shape of X_val_padded: {X_val_padded.shape}")
print(f"Shape of X_test_padded: {X_test_padded.shape}")
print(f"X_train dtype: {X_train_padded.dtype}; X_val dtype: {X_val_padded.dtype}; X_test dtype: {X_test_padded.dtype}")

# Convert labels to one-hot encoding
y_train_one_hot = keras.utils.to_categorical(y_train_videos, num_classes=NUM_CLASSES)
y_val_one_hot = keras.utils.to_categorical(y_val_videos, num_classes=NUM_CLASSES)
y_test_one_hot = keras.utils.to_categorical(y_test_videos, num_classes=NUM_CLASSES)

print(f"Shape of y_train_one_hot: {y_train_one_hot.shape}")
print(f"Shape of y_val_one_hot: {y_val_one_hot.shape}")
print(f"Shape of y_test_one_hot: {y_test_one_hot.shape}")
print(f"y_train dtype: {y_train_one_hot.dtype}; y_val dtype: {y_val_one_hot.dtype}; y_test dtype: {y_test_one_hot.dtype}")

# Force datatype to float 32 to ensure Object is not one of the data types
import numpy as np

# 1. Force X data to float32 and normalize (0-1 range)
X_train_padded = np.array(X_train_padded, dtype='float32')
X_val_padded = np.array(X_val_padded, dtype='float32')
X_test_padded = np.array(X_test_padded, dtype='float32')

# 2. Convert y data to float32 (optional but recommended for consistency)
y_train_one_hot = y_train_one_hot.astype('float32')
y_val_one_hot = y_val_one_hot.astype('float32')
y_test_one_hot = y_test_one_hot.astype('float32')

print("Conversion complete!")
print(f"X_train dtype: {X_train_padded.dtype}; X_val dtype: {X_val_padded.dtype}; X_test dtype: {X_test_padded.dtype}")
print(f"y_train dtype: {y_train_one_hot.dtype}; y_val dtype: {y_val_one_hot.dtype}; y_test dtype: {y_test_one_hot.dtype}")


Num of Classes : 7
Original dimensions per frame: H=180, W=320, C=3
Preprocessing training videos...
Preprocessing validation videos...
Preprocessing test videos...
Shape of X_train_padded: (254, 13, 180, 320, 3)
Shape of X_val_padded: (55, 13, 180, 320, 3)
Shape of X_test_padded: (55, 13, 180, 320, 3)
X_train dtype: float32; X_val dtype: float32; X_test dtype: float32
Shape of y_train_one_hot: (254, 7)
Shape of y_val_one_hot: (55, 7)
Shape of y_test_one_hot: (55, 7)
y_train dtype: float64; y_val dtype: float64; y_test dtype: float64
Conversion complete!
X_train dtype: float32; X_val dtype: float32; X_test dtype: float32
y_train dtype: float32; y_val dtype: float32; y_test dtype: float32


## Build the NN models
### Option 1: Build the 3D-CNN model from scratch
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Input(shape=(SEQUENCE_LENGTH, f_HEIGHT, f_WIDTH, f_CHANNELS)),

    layers.Conv3D(filters=16, kernel_size=(3, 3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPool3D(pool_size=(1, 2, 2)),

    layers.Conv3D(filters=32, kernel_size=(3, 3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPool3D(pool_size=(1, 2, 2)),

    layers.Conv3D(filters=64, kernel_size=(3, 3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPool3D(pool_size=(1, 2, 2)),

    #layers.Flatten(),
    layers.GlobalAveragePooling3D(),
    #layers.Dense(units=128, activation='relu'),
    layers.Dense(units=256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(units=NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy']
)

model.summary()

### Use LSTM model and leverage existing ResNet to achieve good performance

In [5]:
#Leverage ResNet50 and add LSTM
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import ResNet50, resnet50
from tensorflow.keras import layers, models, optimizers

# 1. Load the pre-trained 'brain'
video_shape = (SEQUENCE_LENGTH, HEIGHT, WIDTH, 3) # Based on current frame setup
resnet_base = ResNet50(weights='imagenet', include_top=False, input_shape=(HEIGHT, WIDTH, 3))
resnet_base.trainable = False  # Keep existing knowledge frozen

# 2. Build the Hybrid Model (LRCN)
model = models.Sequential([
    layers.Input(shape=video_shape),

    layers.Rescaling(255.0),
    # This automatically scales your 0-1 or 0-255 data for ResNet
    layers.TimeDistributed(layers.Lambda(resnet50.preprocess_input)),

    # Extract spatial features from each frame
    layers.TimeDistributed(resnet_base),
    layers.TimeDistributed(layers.GlobalAveragePooling2D()),

    # Learn the temporal 'swing' motion
    layers.LSTM(64, return_sequences=False, dropout=0.2),

    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.0002), # Slower LR for Transfer Learning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


# Model Training
# Learning rate scheduler to start with fast learning rate
#  and reduce lr when validation loss is not decreasing
lr_reducer = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.25,
    patience=3,
    min_lr=1e-6
)

# Define Early Stopping
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',      # Watch the validation loss
    patience=5,              # Number of epochs to wait for improvement
    mode='min',              # We want the loss to be at its minimum
    restore_best_weights=True, # VERY IMPORTANT: Reverts the model to the best epoch
    verbose=1                # Prints a message when training stops
)

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='best_tennis_model.keras',
    monitor='val_loss',
    save_best_only=True,  # Only saves when val_loss improves
    verbose=1
)

# Train the model in Phase 1
# resnet_base.trainable = False to Keep existing knowledge frozen
BATCH_SIZE = 4
EPOCHS = 35 # You can adjust this
history = model.fit(
    X_train_padded, y_train_one_hot,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_padded, y_val_one_hot),
    callbacks=[lr_reducer,early_stopping, checkpoint_callback], # Add the callback here
    verbose=1
)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 13, 180, 320,   │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 13, 180, 320,   │             0 │
│ (TimeDistributed)               │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 13, 6, 10,      │    23,587,712 │
│ (TimeDistributed)               │ 2048)                  │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 13, 2048)       │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │       540,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,147,079 (92.11 MB)

 Trainable params: 559,367 (2.13 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

Epoch 1/35
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1587 - loss: 1.9693
Epoch 1: val_loss improved from None to 1.86850, saving model to best_tennis_model.keras

Epoch 1: finished saving model to best_tennis_model.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 157s 2s/step - accuracy: 0.1575 - loss: 1.9621 - val_accuracy: 0.3273 - val_loss: 1.8685 - learning_rate: 2.0000e-04
Epoch 2/35
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1936 - loss: 1.8744
Epoch 2: val_loss improved from 1.86850 to 1.82796, saving model to best_tennis_model.keras

Epoch 2: finished saving model to best_tennis_model.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 94s 1s/step - accuracy: 0.2165 - loss: 1.8825 - val_accuracy: 0.3455 - val_loss: 1.8280 - learning_rate: 2.0000e-04
Epoch 3/35
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.2721 - loss: 1.8169
Epoch 3: val_loss improved from 1.82796 to 1.72334, saving model to best_tennis_model.keras

Epoch 3: finished saving model to best_tennis_model.keras
64/64 ━━━━━━━━

## 4. Evaluation

In [6]:

from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Make predictions on the test set using the trained 3D-CNN model
y_pred_one_hot = model.predict(X_test_padded)
y_pred_classes = np.argmax(y_pred_one_hot, axis=1)
y_true_classes = np.argmax(y_test_one_hot, axis=1) # Use y_test_one_hot from training setup

print("Classification Report:")
print(classification_report(y_true_classes, y_pred_classes, target_names=class_names))

# Print confusion matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_true_classes, y_pred_classes)
print(cm)


2/2 ━━━━━━━━━━━━━━━━━━━━ 46s 22s/step
Classification Report:
              precision    recall  f1-score   support

     b_slice       0.80      1.00      0.89         8
    b_volley       0.88      1.00      0.93         7
    backhand       1.00      0.62      0.77         8
    f_volley       1.00      0.88      0.93         8
    forehand       0.70      0.88      0.78         8
       serve       1.00      0.88      0.93         8
       smash       1.00      1.00      1.00         8

    accuracy                           0.89        55
   macro avg       0.91      0.89      0.89        55
weighted avg       0.91      0.89      0.89        55


Confusion Matrix:
[[8 0 0 0 0 0 0]
 [0 7 0 0 0 0 0]
 [1 0 5 0 2 0 0]
 [0 1 0 7 0 0 0]
 [1 0 0 0 7 0 0]
 [0 0 0 0 1 7 0]
 [0 0 0 0 0 0 8]]


In [7]:
# In case of val_loss "stuck" during phase 1, start Phase 2
# resnet_base.trainable = True to train features specific to tennis <maybe>
# Only the last 20 layers though
resnet_base.trainable = True

# Fine-tune only the last 20 layers
for layer in resnet_base.layers[:-20]:
    layer.trainable = False

# Re-compile with a VERY small learning rate
model.compile(optimizer=optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])


checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='best_unfreeze_tennis_model.keras',
    monitor='val_loss',
    save_best_only=True,  # Only saves when val_loss improves
    verbose=1
)

BATCH_SIZE = 2
EPOCHS = 20
history = model.fit(
    X_train_padded, y_train_one_hot,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_padded, y_val_one_hot),
    callbacks=[lr_reducer,early_stopping, checkpoint_callback], # Add the callback here
    verbose=1
)

Epoch 1/20
127/127 ━━━━━━━━━━━━━━━━━━━━ 0s 922ms/step - accuracy: 0.7341 - loss: 0.8000
Epoch 1: val_loss improved from None to 0.54081, saving model to best_unfreeze_tennis_model.keras

Epoch 1: finished saving model to best_unfreeze_tennis_model.keras
127/127 ━━━━━━━━━━━━━━━━━━━━ 206s 1s/step - accuracy: 0.8071 - loss: 0.5941 - val_accuracy: 0.8545 - val_loss: 0.5408 - learning_rate: 1.0000e-05
Epoch 2/20
127/127 ━━━━━━━━━━━━━━━━━━━━ 0s 937ms/step - accuracy: 0.9122 - loss: 0.4005
Epoch 2: val_loss improved from 0.54081 to 0.52472, saving model to best_unfreeze_tennis_model.keras

Epoch 2: finished saving model to best_unfreeze_tennis_model.keras
127/127 ━━━━━━━━━━━━━━━━━━━━ 138s 1s/step - accuracy: 0.9173 - loss: 0.3740 - val_accuracy: 0.8727 - val_loss: 0.5247 - learning_rate: 1.0000e-05
Epoch 3/20
127/127 ━━━━━━━━━━━━━━━━━━━━ 0s 934ms/step - accuracy: 0.9151 - loss: 0.3102
Epoch 3: val_loss did not improve from 0.52472
127/127 ━━━━━━━━━━━━━━━━━━━━ 137s 1s/step - accuracy: 0.9016 -

In [8]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Make predictions on the test set using the trained 3D-CNN model
y_pred_one_hot = model.predict(X_test_padded)
y_pred_classes = np.argmax(y_pred_one_hot, axis=1)
y_true_classes = np.argmax(y_test_one_hot, axis=1) # Use y_test_one_hot from training setup

print("Classification Report:")
print(classification_report(y_true_classes, y_pred_classes, target_names=class_names))

# Print confusion matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_true_classes, y_pred_classes)
print(cm)


2/2 ━━━━━━━━━━━━━━━━━━━━ 50s 23s/step
Classification Report:
              precision    recall  f1-score   support

     b_slice       0.80      1.00      0.89         8
    b_volley       0.86      0.86      0.86         7
    backhand       0.88      0.88      0.88         8
    f_volley       0.88      0.88      0.88         8
    forehand       1.00      0.75      0.86         8
       serve       1.00      1.00      1.00         8
       smash       1.00      1.00      1.00         8

    accuracy                           0.91        55
   macro avg       0.92      0.91      0.91        55
weighted avg       0.92      0.91      0.91        55


Confusion Matrix:
[[8 0 0 0 0 0 0]
 [0 6 0 1 0 0 0]
 [1 0 7 0 0 0 0]
 [0 1 0 7 0 0 0]
 [1 0 1 0 6 0 0]
 [0 0 0 0 0 8 0]
 [0 0 0 0 0 0 8]]


In [12]:
#Load model and verify validation accuracy
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import resnet50

# Define the custom objects so Keras can rebuild the Lambda layer
custom_dict = {"preprocess_input": resnet50.preprocess_input}

# Load the model with the custom_objects parameter
model = tf.keras.models.load_model(
    'best_unfreeze_tennis_model.keras',
    custom_objects=custom_dict
)

print("Model loaded successfully!")
# Now evaluate again - you should see the 87.27% accuracy
val_loss, val_acc = model.evaluate(X_val_padded, y_val_one_hot)
print(f"Verified Validation Accuracy: {val_acc}")

Model loaded successfully!
2/2 ━━━━━━━━━━━━━━━━━━━━ 39s 6s/step - accuracy: 0.8727 - loss: 0.5020
Verified Validation Accuracy: 0.8727272748947144


In [14]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Make predictions on the test set using the trained 3D-CNN model
y_pred_one_hot = model.predict(X_test_padded)
y_pred_classes = np.argmax(y_pred_one_hot, axis=1)
y_true_classes = np.argmax(y_test_one_hot, axis=1) # Use y_test_one_hot from training setup

print("Classification Report:")
print(classification_report(y_true_classes, y_pred_classes, target_names=class_names))

# Print confusion matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_true_classes, y_pred_classes)
print(cm)


1/2 ━━━━━━━━━━━━━━━━━━━━ 24s 25s/stepWARNING:tensorflow:6 out of the last 6 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x000002D44073A700> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
2/2 ━━━━━━━━━━━━━━━━━━━━ 46s 21s/step
Classification Report:
              precision    recall  f1-score   support

     b_slice       0.89      1.00      0.94         8
    b_volley       0.88      1.00      0.93         7
    backhand 